<a href="https://colab.research.google.com/github/Aisyaaswy/Pemrograman_Mesin/blob/main/jobsheet_6/HNSW/praktikum_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PRAKTIKUM 4**
Percobaan kali ini kita akan melihat perbedaan ketiga model yang telah kita bahas dan bandingkan hasilnya.


In [6]:
!pip install annoy faiss-cpu hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 68.7 MB/s eta 0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520034 sha256=10b5c48d04c72fefff7bf1ea63c8394abdbf59f4d5ee77694212b67aa12f42aa
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
Successfully built annoy


In [7]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

# ===============================
# 1. Buat dataset 1 juta data 5D
# ===============================
n_data = 1_000_000   # bisa coba 100_000 dulu jika RAM terbatas
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10

# ===============================
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

# ===============================
# 3. FAISS (Flat Index)
# ===============================
print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")

# ===============================
# 4. HNSW (hnswlib)
# ===============================
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")


=== Annoy ===
Build time: 22.144516944885254 detik
Query time: 0.00021266937255859375 detik
Neighbors: [798808, 253512, 797205, 885447, 599858] ...

=== FAISS (IndexFlatL2) ===
Build time: 0.016811370849609375 detik
Query time: 0.013530731201171875 detik
Neighbors: [798808 253512 797205 885447 599858] ...

=== HNSW (hnswlib) ===
Build time: 175.48920035362244 detik
Query time: 0.0003654956817626953 detik
Neighbors: [798808 253512 797205 885447 599858] ...


Lakukan percobaan pada metric distance inner product. catat hasilnya pada tabel yang anda buat sendiri seperti pada praktikum 1.

In [11]:
import numpy as np
import time
import pandas as pd
from annoy import AnnoyIndex
import faiss
import hnswlib

# ===============================
# 1. Konfigurasi dataset
# ===============================
np.random.seed(42)

n_data = 1_000_000
dim = 5
k = 10

# Buat dataset
X = np.random.random((n_data, dim)).astype(np.float32)

# Normalisasi L2 agar Inner Product setara dengan Cosine Similarity
faiss.normalize_L2(X)

# Buat query
query = np.random.random((1, dim)).astype(np.float32)
faiss.normalize_L2(query)

# Menampung hasil eksperimen
hasil_eksperimen = []

# ===============================
# 2. Annoy (Angular)
# ===============================
ann_index = AnnoyIndex(dim, 'angular')

start = time.time()

for i in range(n_data):
    ann_index.add_item(i, X[i])

n_trees = 10
ann_index.build(n_trees)

build_time = time.time() - start

start = time.time()

ann_neighbors = ann_index.get_nns_by_vector(
    query[0],
    k,
    include_distances=True
)

query_time = time.time() - start

# Ambil 5 indeks terdekat
ann_idx = ann_neighbors[0][:5]
str_idx = "[" + " ".join(map(str, ann_idx)) + "]"

hasil_eksperimen.append({
    'Metode': 'Annoy',
    'Distance Metrics': 'Angular',
    'Parameter': f'{n_trees} Trees',
    'Jumlah data': n_data,
    'Dimensi': dim,
    'Hasil Index terdekat (Top 5)': str_idx,
    'Waktu Build (s)': f'{build_time:.4f}',
    'Waktu Query (s)': f'{query_time:.4f}'
})

# ===============================
# 3. FAISS (IndexFlatIP)
# ===============================
faiss_index = faiss.IndexFlatIP(dim)

start = time.time()

faiss_index.add(X)

build_time = time.time() - start

start = time.time()

distances, indices = faiss_index.search(query, k)

query_time = time.time() - start

# Ambil 5 indeks terdekat
faiss_idx = indices[0][:5]
str_idx = "[" + " ".join(map(str, faiss_idx)) + "]"

hasil_eksperimen.append({
    'Metode': 'FAISS',
    'Distance Metrics': 'Inner Product',
    'Parameter': 'IndexFlatIP',
    'Jumlah data': n_data,
    'Dimensi': dim,
    'Hasil Index terdekat (Top 5)': str_idx,
    'Waktu Build (s)': f'{build_time:.4f}',
    'Waktu Query (s)': f'{query_time:.4f}'
})

# ===============================
# 4. HNSW (hnswlib)
# ===============================
hnsw_index = hnswlib.Index(space='ip', dim=dim)

start = time.time()

hnsw_index.init_index(
    max_elements=n_data,
    ef_construction=200,
    M=16
)

hnsw_index.add_items(X)

build_time = time.time() - start

# Mengatur parameter pencarian
ef_search = 50
hnsw_index.set_ef(ef_search)

start = time.time()

labels, distances = hnsw_index.knn_query(query, k=k)

query_time = time.time() - start

# Ambil 5 indeks terdekat
hnsw_idx = labels[0][:5]
str_idx = "[" + " ".join(map(str, hnsw_idx)) + "]"

hasil_eksperimen.append({
    'Metode': 'HNSW',
    'Distance Metrics': 'Inner Product',
    'Parameter': f'M=16, ef_construction=200, ef={ef_search}',
    'Jumlah data': n_data,
    'Dimensi': dim,
    'Hasil Index terdekat (Top 5)': str_idx,
    'Waktu Build (s)': f'{build_time:.4f}',
    'Waktu Query (s)': f'{query_time:.4f}'
})

# ===============================
# 5. Tampilkan tabel hasil
# ===============================
df_hasil = pd.DataFrame(hasil_eksperimen)

display(df_hasil)


,Metode,Distance Metrics,Parameter,Jumlah data,Dimensi,Hasil Index terdekat (Top 5),Waktu Build (s),Waktu Query (s)
0,Annoy,Angular,10 Trees,1000000,5,[653774 135588 561017 901095 211473],28.9867,0.0003
1,FAISS,Inner Product,IndexFlatIP,1000000,5,[653774 135588 561017 901095 211473],0.0081,0.0036
2,HNSW,Inner Product,"M=16, ef_construction=200, ef=50",1000000,5,[653774 135588 561017 901095 211473],168.3507,0.0003
